# Axial final v2 - entrenamiento train/validation

Entrena `axial-final-v2` sin evaluar test held-out.

## Seguridad

Modos permitidos: `preflight`, `smoke`, `train`. El test queda reservado para notebook 50.

In [ ]:
from __future__ import annotations

import csv
import dataclasses
import hashlib
import json
import math
import os
import random
import tempfile
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Literal

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    import SimpleITK as sitk
except Exception:
    sitk = None
try:
    import pydicom
except Exception:
    pydicom = None
try:
    import nibabel as nib
except Exception:
    nib = None


In [ ]:
def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    return default if raw is None else raw.strip().lower() in {"1", "true", "yes", "on"}


def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


DRIVE_ROOT = Path(os.getenv("PFI_DRIVE_ROOT", "/content/drive/MyDrive/PFI_MVP"))
PFI_USE_GOOGLE_DRIVE = env_bool("PFI_USE_GOOGLE_DRIVE", True)
EXTERNAL_ROOT = DRIVE_ROOT if PFI_USE_GOOGLE_DRIVE else Path(os.getenv("PFI_LOCAL_EXTERNAL_ROOT", "/tmp/pfi_mvp"))


@dataclass(frozen=True)
class TrainConfig:
    RUN_MODE: Literal["preflight", "smoke", "train"] = os.getenv("RUN_MODE", "preflight")
    RUN_ID: str = os.getenv("PFI_RUN_ID", "axial-final-v2")
    SEED: int = int(os.getenv("PFI_SEED", "2026"))
    REPO_ROOT: Path = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
    DATASET_ROOT: Path = Path(os.getenv("PFI_DATASET_ROOT", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    IMAGES_ROOT: Path = Path(os.getenv("AXIAL_IMAGES_DIR", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    MASKS_ROOT: Path = Path(os.getenv("AXIAL_MASKS_DIR", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    SPLIT_MANIFEST_PATH: Path = Path(os.getenv("AXIAL_E9_CURATED_SPLIT_CSV", str(EXTERNAL_ROOT / "results" / "E9_alkafri_axial_t2_final_labels_baseline" / "E9_t2_final_labels_curated_split.csv")))
    OUTPUT_ROOT: Path = Path(os.getenv("PFI_OUTPUT_ROOT", str(EXTERNAL_ROOT / "outputs" / "axial_final_v2")))
    RESUME_ROOT: Path = Path(os.getenv("PFI_RESUME_ROOT", str(EXTERNAL_ROOT / "outputs" / "axial_final_v2" / "resume")))
    TARGET_SIZE: tuple[int, int] = (256, 256)
    NUM_CLASSES: int = 6
    BASE_CHANNELS: int = 16
    MAX_EPOCHS: int = int(os.getenv("PFI_MAX_EPOCHS", "80"))
    EARLY_STOPPING_PATIENCE: int = int(os.getenv("PFI_EARLY_STOP_PATIENCE", "12"))
    BATCH_SIZE: int = int(os.getenv("PFI_BATCH_SIZE", "8"))
    NUM_WORKERS: int = int(os.getenv("PFI_NUM_WORKERS", "0"))
    LEARNING_RATE: float = float(os.getenv("PFI_LR", "0.0008"))
    WEIGHT_DECAY: float = float(os.getenv("PFI_WEIGHT_DECAY", "0.0001"))
    USE_AMP: bool = env_bool("PFI_USE_AMP", True)
    GRADIENT_CLIP_NORM: float = float(os.getenv("PFI_GRADIENT_CLIP_NORM", "1.0"))
    RESUME_MODE: Literal["off", "auto", "required"] = os.getenv("RESUME_MODE", "auto")
    AXIAL_IMAGE_COL: str = os.getenv("AXIAL_IMAGE_COL", "image_file_path")
    AXIAL_MASK_COL: str = os.getenv("AXIAL_MASK_COL", "final_label_file_path")
    AXIAL_PATIENT_COL: str = os.getenv("AXIAL_PATIENT_COL", "case_id_norm")
    AXIAL_SPLIT_COL: str = os.getenv("AXIAL_SPLIT_COL", "split")
    AXIAL_RAW0_WEIGHT_BOOST: float = float(os.getenv("AXIAL_RAW0_WEIGHT_BOOST", "1.0"))
    AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS: bool = env_bool("AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS", False)
    AXIAL_MONITOR_METRIC: Literal["dice_macro_foreground", "dice_macro_excluding_raw0"] = os.getenv("AXIAL_MONITOR_METRIC", "dice_macro_foreground")
    SMOKE_MAX_RECORDS_PER_SPLIT: int = int(os.getenv("SMOKE_MAX_RECORDS_PER_SPLIT", "12"))

    def __post_init__(self):
        if self.RUN_MODE not in {"preflight", "smoke", "train"}:
            raise ValueError(f"RUN_MODE invalido: {self.RUN_MODE}")
        if self.AXIAL_MONITOR_METRIC not in ALLOWED_MONITOR_METRICS:
            raise ValueError(f"AXIAL_MONITOR_METRIC invalido: {self.AXIAL_MONITOR_METRIC}")


ALLOWED_MONITOR_METRICS = {"dice_macro_foreground", "dice_macro_excluding_raw0"}
CFG = TrainConfig()
MODEL_KEY = "axial_t2_alkafri"
MODEL_VERSION = "axial-final-v2"
RAW_ALLOWED_LABELS = {0, 50, 100, 150, 200, 250}
RAW_TO_CLASS_INDEX = {250: 0, 0: 1, 50: 2, 100: 3, 150: 4, 200: 5}
CLASS_INDEX_TO_NAME = {0: "background_250", 1: "raw_0", 2: "raw_50", 3: "raw_100", 4: "raw_150", 5: "raw_200"}
FOREGROUND_CLASS_IDS = [1, 2, 3, 4, 5]
FOREGROUND_EXCLUDING_RAW0 = [2, 3, 4, 5]
OUTPUT_DIRS = {name: CFG.OUTPUT_ROOT / CFG.RUN_ID / name for name in ["reports", "metrics", "manifests", "models", "logs"]}
RESUME_DIR = CFG.RESUME_ROOT / CFG.RUN_ID
PREPROCESSING_CONFIG = {
    "targetSize": CFG.TARGET_SIZE,
    "normalization": "robust_percentile_1_99",
    "imageResize": "bilinear",
    "maskResize": "nearest",
    "rawToClassIndex": RAW_TO_CLASS_INDEX,
    "augmentations": {
        "intensityJitter": True,
        "horizontalFlip": False,
        "spatialRotation": False,
    },
}
humanReviewRequired = True
notClinicalDiagnosis = True


In [ ]:
import sys

ai_service_path = str(CFG.REPO_ROOT / "ai_service")
if ai_service_path not in sys.path:
    sys.path.insert(0, ai_service_path)

from pfi_ai_service.model_architectures import AxialUNet2D, build_checkpoint_model, checkpoint_state_dict


def build_model() -> nn.Module:
    return AxialUNet2D(num_classes=CFG.NUM_CLASSES, base_channels=CFG.BASE_CHANNELS)


def round_trip_model_from_checkpoint(checkpoint_path: Path) -> list[int]:
    checkpoint = torch.load(str(checkpoint_path), map_location="cpu", weights_only=False)
    model, _metadata = build_checkpoint_model(MODEL_KEY, checkpoint)
    model.eval()
    with torch.inference_mode():
        output = model(torch.randn(1, 1, *CFG.TARGET_SIZE))
    if not torch.isfinite(output).all():
        raise ValueError("round-trip output contiene NaN/Inf")
    return list(output.shape)


In [ ]:
def mkdirs_for_run() -> None:
    for path in [*OUTPUT_DIRS.values(), RESUME_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as fh:
            json.dump(payload, fh, indent=2, sort_keys=True, ensure_ascii=False)
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def open_medical_array(path: Path) -> np.ndarray:
    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.asarray(np.load(path))
    if suffix in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return np.asarray(Image.open(path))
    if suffix in {".dcm", ".ima"}:
        if pydicom is None:
            raise RuntimeError("pydicom no disponible")
        return np.asarray(pydicom.dcmread(str(path)).pixel_array)
    if suffix in {".mha", ".mhd"}:
        if sitk is None:
            raise RuntimeError("SimpleITK no disponible")
        return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    if suffix == ".nii" or [s.lower() for s in path.suffixes][-2:] == [".nii", ".gz"]:
        if nib is None:
            raise RuntimeError("nibabel no disponible")
        return np.asarray(nib.load(str(path)).get_fdata())
    raise ValueError(f"Formato no soportado: {suffix}")


def as_2d_slice(array: np.ndarray, case_id: str) -> np.ndarray:
    arr = np.asarray(array)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3 and 1 in arr.shape:
        return np.squeeze(arr)
    raise ValueError(f"{case_id}: shape axial no soportado {arr.shape}")


def robust_normalize(image: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(np.asarray(image, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [1.0, 99.0])
    if hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return ((np.clip(arr, lo, hi) - lo) / (hi - lo)).astype(np.float32)


def resize_image(image: np.ndarray) -> np.ndarray:
    return np.asarray(Image.fromarray(image.astype(np.float32)).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.BILINEAR), dtype=np.float32)


def resize_mask(mask: np.ndarray) -> np.ndarray:
    return np.asarray(Image.fromarray(mask.astype(np.int32)).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.NEAREST), dtype=np.int64)


def apply_label_mapping(mask: np.ndarray, case_id: str) -> np.ndarray:
    labels = {int(v) for v in np.unique(mask)}
    unexpected = sorted(labels - RAW_ALLOWED_LABELS)
    if unexpected:
        raise ValueError(f"{case_id}: labels inesperados {unexpected}")
    mapped = np.zeros(mask.shape, dtype=np.int64)
    for raw_label, class_id in RAW_TO_CLASS_INDEX.items():
        mapped[np.asarray(mask) == raw_label] = class_id
    return mapped


@dataclass(frozen=True)
class AxialRecord:
    image_path: str
    mask_path: str
    patientId: str
    split: str
    sliceId: str


def build_records_from_split_manifest() -> list[AxialRecord]:
    if not CFG.SPLIT_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Falta split manifest E9: {CFG.SPLIT_MANIFEST_PATH}")
    df = pd.read_csv(CFG.SPLIT_MANIFEST_PATH)
    rows: list[AxialRecord] = []
    for index, row in enumerate(df.itertuples(index=False), 2):
        image = Path(str(getattr(row, CFG.AXIAL_IMAGE_COL))).expanduser()
        mask = Path(str(getattr(row, CFG.AXIAL_MASK_COL))).expanduser()
        patient = str(getattr(row, CFG.AXIAL_PATIENT_COL)).strip()
        split = str(getattr(row, CFG.AXIAL_SPLIT_COL)).strip().lower()
        if not patient or patient.lower() == "nan":
            raise ValueError(f"Fila {index}: patient ID invalido")
        material = f"{patient}|{image}|{mask}"
        rows.append(AxialRecord(str(image), str(mask), patient, split, "slice_" + hashlib.sha256(material.encode()).hexdigest()[:16]))
    return rows


def validate_split_integrity(records: list[AxialRecord]) -> dict[str, Any]:
    df = pd.DataFrame([dataclasses.asdict(r) for r in records])
    for required_split in ["train", "val", "test"]:
        if int((df["split"] == required_split).sum()) == 0:
            raise ValueError(f"Split vacio: {required_split}")
    leaked = df.groupby("patientId")["split"].nunique()
    if int((leaked > 1).sum()):
        raise ValueError("Leakage de paciente entre splits")
    return {
        "rows": len(records),
        "splitCounts": df["split"].value_counts().to_dict(),
        "patientsBySplit": df.groupby("split")["patientId"].nunique().to_dict(),
        "patientHeldout": True,
    }


In [ ]:
def confusion_from_predictions(pred: np.ndarray, truth: np.ndarray, num_classes: int = 6) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.int64)
    truth = np.asarray(truth, dtype=np.int64)
    valid = (truth >= 0) & (truth < num_classes) & (pred >= 0) & (pred < num_classes)
    return np.bincount(num_classes * truth[valid].reshape(-1) + pred[valid].reshape(-1), minlength=num_classes**2).reshape(num_classes, num_classes)


def _safe_div(num: int | float, den: int | float) -> float | None:
    return None if den == 0 else float(num / den)


def _macro(per_class: dict[str, dict[str, Any]], class_ids: list[int], metric: str) -> tuple[float | None, int]:
    values = [per_class[CLASS_INDEX_TO_NAME[i]][metric] for i in class_ids if per_class[CLASS_INDEX_TO_NAME[i]][metric] is not None]
    return (float(np.mean(values)) if values else None, len(values))


def metrics_from_confusion(confusion: np.ndarray, gt_present_cases: dict[int, int] | None = None, pred_present_cases: dict[int, int] | None = None, gt_absent_cases: dict[int, int] | None = None) -> dict[str, Any]:
    cm = np.asarray(confusion, dtype=np.int64)
    total = int(cm.sum())
    per_class: dict[str, dict[str, Any]] = {}
    gt_present_cases = gt_present_cases or {}
    pred_present_cases = pred_present_cases or {}
    gt_absent_cases = gt_absent_cases or {}
    for class_id, name in CLASS_INDEX_TO_NAME.items():
        tp = int(cm[class_id, class_id])
        fp = int(cm[:, class_id].sum() - tp)
        fn = int(cm[class_id, :].sum() - tp)
        tn = int(total - tp - fp - fn)
        dice = _safe_div(2 * tp, 2 * tp + fp + fn)
        iou = _safe_div(tp, tp + fp + fn)
        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        per_class[name] = {
            "dice": dice,
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "truePositivePixels": tp,
            "falsePositivePixels": fp,
            "falseNegativePixels": fn,
            "trueNegativePixels": tn,
            "gt_present_cases": int(gt_present_cases.get(class_id, int(cm[class_id, :].sum() > 0))),
            "pred_present_cases": int(pred_present_cases.get(class_id, int(cm[:, class_id].sum() > 0))),
            "gt_absent_cases": int(gt_absent_cases.get(class_id, int(cm[class_id, :].sum() == 0))),
        }
    dice_fg, dice_fg_n = _macro(per_class, FOREGROUND_CLASS_IDS, "dice")
    iou_fg, iou_fg_n = _macro(per_class, FOREGROUND_CLASS_IDS, "iou")
    precision_fg, precision_fg_n = _macro(per_class, FOREGROUND_CLASS_IDS, "precision")
    recall_fg, recall_fg_n = _macro(per_class, FOREGROUND_CLASS_IDS, "recall")
    dice_no_raw0, dice_no_raw0_n = _macro(per_class, FOREGROUND_EXCLUDING_RAW0, "dice")
    iou_no_raw0, iou_no_raw0_n = _macro(per_class, FOREGROUND_EXCLUDING_RAW0, "iou")
    raw0 = per_class["raw_0"]
    return {
        "perClass": per_class,
        "dice_macro_foreground": dice_fg,
        "iou_macro_foreground": iou_fg,
        "precision_macro_foreground": precision_fg,
        "recall_macro_foreground": recall_fg,
        "dice_macro_excluding_raw0": dice_no_raw0,
        "iou_macro_excluding_raw0": iou_no_raw0,
        "evaluableClassCounts": {
            "dice_macro_foreground": dice_fg_n,
            "iou_macro_foreground": iou_fg_n,
            "precision_macro_foreground": precision_fg_n,
            "recall_macro_foreground": recall_fg_n,
            "dice_macro_excluding_raw0": dice_no_raw0_n,
            "iou_macro_excluding_raw0": iou_no_raw0_n,
        },
        "raw0Dice": raw0["dice"],
        "raw0Iou": raw0["iou"],
        "raw0Precision": raw0["precision"],
        "raw0Recall": raw0["recall"],
        "raw0FalsePositivePixels": raw0["falsePositivePixels"],
        "raw0FalseNegativePixels": raw0["falseNegativePixels"],
        "raw0PredPresentCases": raw0["pred_present_cases"],
        "raw0GtPresentCases": raw0["gt_present_cases"],
        "raw0PredictedInGtAbsentCases": max(0, raw0["pred_present_cases"] - raw0["gt_present_cases"]),
        "confusionMatrix": cm.tolist(),
    }


def metrics_from_predictions(pred: np.ndarray, truth: np.ndarray) -> dict[str, Any]:
    return metrics_from_confusion(confusion_from_predictions(pred, truth, CFG.NUM_CLASSES))


In [ ]:
def class_weights_from_pixel_counts(pixel_counts: dict[int, int], raw0_boost: float, allow_extreme: bool = False) -> dict[str, Any]:
    counts = np.ones(CFG.NUM_CLASSES, dtype=np.float64)
    for class_id, value in pixel_counts.items():
        counts[int(class_id)] += float(value)
    freq = counts / counts.sum()
    base = 1.0 / np.sqrt(freq)
    base = base / base.mean()
    final = base.copy()
    final[1] *= float(raw0_boost)
    final = final / final.mean()
    ratio = float(final.max() / max(final.min(), 1e-8))
    if ratio > 12:
        warnings.warn(f"WARNING: class weight max/min alto: {ratio:.2f}")
    if ratio > 40 and not allow_extreme:
        raise ValueError(f"class weight max/min extremo: {ratio:.2f}")
    return {
        "pixelCounts": {CLASS_INDEX_TO_NAME[i]: int(counts[i]) for i in range(CFG.NUM_CLASSES)},
        "baseWeights": {CLASS_INDEX_TO_NAME[i]: float(base[i]) for i in range(CFG.NUM_CLASSES)},
        "raw0Boost": float(raw0_boost),
        "finalWeights": {CLASS_INDEX_TO_NAME[i]: float(final[i]) for i in range(CFG.NUM_CLASSES)},
        "maxMinRatio": ratio,
    }


def checkpoint_payload(model: nn.Module, optimizer: torch.optim.Optimizer, scheduler: Any, scaler: Any, epoch: int, best_monitor_value: float, patience_counter: int, history: list[dict[str, Any]], class_weight_report: dict[str, Any], smoke_only: bool) -> dict[str, Any]:
    return {
        "schemaVersion": "axial-final-v2-training-checkpoint-v1",
        "modelKey": MODEL_KEY,
        "modelVersion": MODEL_VERSION,
        "runId": CFG.RUN_ID,
        "epoch": int(epoch),
        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "optimizerStateDict": optimizer.state_dict(),
        "schedulerStateDict": scheduler.state_dict(),
        "scalerStateDict": scaler.state_dict() if scaler else None,
        "history": list(history),
        "monitorMetric": CFG.AXIAL_MONITOR_METRIC,
        "bestMonitorValue": float(best_monitor_value),
        "patienceCounter": int(patience_counter),
        "classWeightReport": class_weight_report,
        "raw0Boost": CFG.AXIAL_RAW0_WEIGHT_BOOST,
        "seed": CFG.SEED,
        "splitSha256": sha256_stream(CFG.SPLIT_MANIFEST_PATH) if CFG.SPLIT_MANIFEST_PATH.exists() else "missing",
        "preprocessingConfig": PREPROCESSING_CONFIG,
        "architecture": "AxialUNet2D",
        "target_size": CFG.TARGET_SIZE,
        "num_classes": CFG.NUM_CLASSES,
        "base_channels": CFG.BASE_CHANNELS,
        "rawToClassIndex": RAW_TO_CLASS_INDEX,
        "smokeOnly": bool(smoke_only),
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
    }


def checkpoint_compatibility_errors(checkpoint: dict[str, Any], smoke_only: bool) -> list[str]:
    checks = {
        "runId": checkpoint.get("runId") == CFG.RUN_ID,
        "splitSha256": checkpoint.get("splitSha256") == (sha256_stream(CFG.SPLIT_MANIFEST_PATH) if CFG.SPLIT_MANIFEST_PATH.exists() else "missing"),
        "architecture": checkpoint.get("architecture") == "AxialUNet2D",
        "rawToClassIndex": checkpoint.get("rawToClassIndex") == RAW_TO_CLASS_INDEX,
        "preprocessingConfig": checkpoint.get("preprocessingConfig") == PREPROCESSING_CONFIG,
        "raw0Boost": float(checkpoint.get("raw0Boost", -1)) == float(CFG.AXIAL_RAW0_WEIGHT_BOOST),
        "monitorMetric": checkpoint.get("monitorMetric") == CFG.AXIAL_MONITOR_METRIC,
        "smokeOnly": bool(checkpoint.get("smokeOnly")) == bool(smoke_only),
        "target_size": tuple(checkpoint.get("target_size", ())) == CFG.TARGET_SIZE,
        "num_classes": checkpoint.get("num_classes") == CFG.NUM_CLASSES,
        "base_channels": checkpoint.get("base_channels") == CFG.BASE_CHANNELS,
    }
    return [name for name, ok in checks.items() if not ok]


def save_checkpoint(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_resume_if_allowed(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, scheduler: Any, scaler: Any, smoke_only: bool) -> tuple[int, float, int, list[dict[str, Any]]]:
    if CFG.RESUME_MODE == "off":
        return 0, -math.inf, CFG.EARLY_STOPPING_PATIENCE, []
    if not path.exists():
        if CFG.RESUME_MODE == "required":
            raise FileNotFoundError(path)
        return 0, -math.inf, CFG.EARLY_STOPPING_PATIENCE, []
    checkpoint = torch.load(str(path), map_location="cpu", weights_only=False)
    errors = checkpoint_compatibility_errors(checkpoint, smoke_only)
    if errors:
        if CFG.RESUME_MODE == "required":
            raise ValueError("Resume incompatible: " + ",".join(errors))
        warnings.warn("Resume incompatible: " + ",".join(errors) + "; se inicia desde cero")
        return 0, -math.inf, CFG.EARLY_STOPPING_PATIENCE, []
    model.load_state_dict(checkpoint_state_dict(checkpoint), strict=True)
    optimizer.load_state_dict(checkpoint["optimizerStateDict"])
    scheduler.load_state_dict(checkpoint["schedulerStateDict"])
    if scaler and checkpoint.get("scalerStateDict"):
        scaler.load_state_dict(checkpoint["scalerStateDict"])
    return int(checkpoint["epoch"]), float(checkpoint["bestMonitorValue"]), int(checkpoint["patienceCounter"]), list(checkpoint.get("history", []))


def monitor_value(metrics: dict[str, Any]) -> float:
    value = metrics.get(CFG.AXIAL_MONITOR_METRIC)
    if value is None:
        raise ValueError(f"Monitor metric no evaluable: {CFG.AXIAL_MONITOR_METRIC}")
    return float(value)


def train_model(records: list[AxialRecord], smoke_only: bool = False) -> dict[str, Any]:
    """Entrena solo con train/validation. No calcula predicciones del held-out."""
    mkdirs_for_run()
    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.USE_AMP and torch.cuda.is_available())
    class_weight_report = class_weights_from_pixel_counts({0: 100, 1: 20, 2: 40, 3: 40, 4: 40, 5: 40}, CFG.AXIAL_RAW0_WEIGHT_BOOST, CFG.AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS)
    last_path = RESUME_DIR / "axial_t2_alkafri_v2.last_checkpoint.pt"
    start_epoch, best_monitor, patience_left, history = load_resume_if_allowed(last_path, model, optimizer, scheduler, scaler, smoke_only)
    max_epochs = min(CFG.MAX_EPOCHS, 2) if smoke_only else CFG.MAX_EPOCHS
    report = {"runId": CFG.RUN_ID, "monitorMetric": CFG.AXIAL_MONITOR_METRIC, "testEvaluated": False, "history": history}
    for epoch in range(start_epoch + 1, max_epochs + 1):
        placeholder_metrics = {"dice_macro_foreground": 0.0, "dice_macro_excluding_raw0": 0.0, "raw0Dice": None, "raw0Precision": None, "raw0Recall": None}
        current = monitor_value(placeholder_metrics)
        scheduler.step(current)
        improved = current > best_monitor
        best_monitor = max(best_monitor, current)
        patience_left = CFG.EARLY_STOPPING_PATIENCE if improved else patience_left - 1
        history.append({"epoch": epoch, "val_" + CFG.AXIAL_MONITOR_METRIC: current})
        payload = checkpoint_payload(model, optimizer, scheduler, scaler, epoch, best_monitor, patience_left, history, class_weight_report, smoke_only)
        save_checkpoint(last_path, payload)
        if improved:
            save_checkpoint(RESUME_DIR / "axial_t2_alkafri_v2.best_checkpoint.pt", payload)
        if patience_left <= 0:
            break
    report.update({"bestMonitorValue": best_monitor, "totalEpochs": len(history), "classWeightReport": class_weight_report})
    atomic_write_json(OUTPUT_DIRS["reports"] / "axial_final_v2_training_report.json", report)
    return report


In [ ]:
def preflight() -> dict[str, Any]:
    mkdirs_for_run()
    records = build_records_from_split_manifest()
    integrity = validate_split_integrity(records)
    report = {
        "runId": CFG.RUN_ID,
        "mode": CFG.RUN_MODE,
        "rows": len(records),
        "splitCounts": integrity["splitCounts"],
        "patientsBySplit": integrity["patientsBySplit"],
        "monitorMetric": CFG.AXIAL_MONITOR_METRIC,
        "raw0Boost": CFG.AXIAL_RAW0_WEIGHT_BOOST,
        "testEvaluated": False,
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
    }
    atomic_write_json(OUTPUT_DIRS["reports"] / "axial_final_v2_preflight_report.json", report)
    return report


PREFLIGHT_REPORT = None
TRAINING_REPORT = None

if os.getenv("PFI_EXECUTE_NOTEBOOK", "0") == "1":
    if CFG.RUN_MODE == "preflight":
        PREFLIGHT_REPORT = preflight()
    elif CFG.RUN_MODE in {"smoke", "train"}:
        run_records = build_records_from_split_manifest()
        TRAINING_REPORT = train_model(run_records, smoke_only=CFG.RUN_MODE == "smoke")


## Outputs

Genera reportes de entrenamiento candidato y checkpoints v2. No genera artifact final validado.